# Homework — Reproducible NLP Pipeline in a Container (Slim Version)

**Course:** Machine Learning, NLP and Toolchain
**Time budget:** 4-6 hours over one week
**Hand in:** Git repo URL + the completed notebook + `reflection.md`

## What you will build

A small, reproducible NLP comparison: classical baseline (TF-IDF + LogReg)
vs. a pretrained transformer (zero-shot HF `pipeline`), packaged so a
stranger can `docker build` and `docker run` and see the results print.

## How to use this notebook

- Fill in every `# TODO` block. Don't delete the comments — they hint at what's expected.
- Each step builds on the previous one. **Run cells top-to-bottom.**
- Where you see `# HINT: ...`, that's a free clue.
- When you finish, the last cell writes `results/results.csv`. That file is
  what the Docker container will reproduce.

## Submission checklist (slide 11)

By the time you submit, all of these must be true:

- [ ] `docker build -t nlp-hw .` succeeds on a fresh clone
- [ ] `docker run --rm nlp-hw` prints the comparison table
- [ ] `results/results.csv` exists with both models
- [ ] `requirements.txt` has pinned versions
- [ ] `.gitignore` excludes `.venv/`, `__pycache__/`, `data/raw/`, `.env`
- [ ] `README.md` shows the two reproduce commands
- [ ] `reflection.md` answers all three questions, under one page


## Step 1 — Setup

Imports and a single global seed. Use `SEED = 42` everywhere so your run is
reproducible.

In [2]:
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

# TODO: set the random seeds for `random` and `numpy` using SEED = 42
# Both libraries have their own RNG state, so both need seeding.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths — already set up for you
DATA_DIR    = Path("data/raw")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Step 2 — Download the dataset

We use the **SMS Spam Collection** — ~5.5k labelled SMS messages (`ham` /
`spam`). Public, ~470 KB, easy to download.

### Your task

1. Use `urllib.request` to download from one of the GitHub mirrors below.
2. Save the file to `DATA_DIR / "sms.tsv"`.
3. If the first mirror fails, try the next one.
4. Don't re-download if the file already exists (be kind to the mirror).

**HINT:** GitHub's raw URLs occasionally need a `User-Agent` header to
avoid being rate-limited. Use `urllib.request.Request(url, headers={...})`.


In [3]:
import urllib.request
from pathlib import Path

SMS_PATH = DATA_DIR / "sms.tsv"

MIRRORS = [
    "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv",
    "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv",
]

def download_sms(target: Path) -> bool:
    # 1. If `target` already exists and is large enough (>50 KB), return True early
    if target.exists() and target.stat().st_size > 50_000:
        return True

    # 2. Loop over MIRRORS, try each one with a User-Agent header.
    for link in MIRRORS:
        try:
            req = urllib.request.Request(link, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as response:
                target.write_bytes(response.read())
            return True
        except Exception as e:
            print(e)
            continue
    if not target.exists():
        return False
    # 3. Return True on success, False if all mirrors fail.


download_sms(SMS_PATH)


True

### Load it into a DataFrame

The file is tab-separated, two columns: `label` (`ham` or `spam`) and `text`.

#### Your task

- Read the TSV into a pandas DataFrame.
- Convert `label` to an integer column `y`: `spam` → 1, `ham` → 0.
- Return a clean DataFrame with columns `text` and `y`.


In [4]:
def load_sms(path: Path = SMS_PATH) -> pd.DataFrame:
    # TODO:
    # 1. Use pd.read_csv with sep="\t", header=None, names=["label", "text"],
    #    encoding="latin-1" (the dataset has some non-UTF-8 characters).
    df = pd.read_csv(path, sep="\t", header=None, names=["label", "text"], encoding="latin-1")

    # 2. Add a column `y` that is 1 for spam, 0 for ham.
    df["y"] = df["label"].map({"ham": 0, "spam": 1})
    
    # 3. Return df[["text", "y"]] with NaNs dropped.
    return df[["text", "y"]].dropna()


df = load_sms()
print(f"Total: {len(df):,} examples")
print(df["y"].value_counts().rename({0:"ham", 1:"spam"}).to_string())
df.head()


Total: 5,572 examples
y
ham     4825
spam     747


,text,y
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


## Step 3 — Train/test split

A stratified split with a fixed seed. **The same split must be used for
both models** — otherwise the comparison is meaningless.

### Your task

Use `sklearn.model_selection.train_test_split` with:
- `test_size = 0.2`
- `random_state = SEED`
- `stratify = ...` so both splits have the same class balance


In [5]:
from sklearn.model_selection import train_test_split

X = np.asarray(df["text"].tolist())
y = np.asarray(df["y"].tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

texts_train, texts_test = X_train, X_test
print(f"train: {len(X_train):,}   test: {len(X_test):,}")


train: 4,457   test: 1,115


## Step 4 — Classical baseline: TF-IDF + Logistic Regression

This is the **floor**. The transformer needs to clearly beat this.

The slide gives you the recipe (slim slide 7, left column). Translate it
into running code below.

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = vec.fit_transform(texts_train)
X_test  = vec.transform(texts_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
```

### Your task

1. Run the slide code above.
2. Time it (`t_classical = ...`).
3. Compute accuracy and macro-F1 on the test set.
4. Print a `classification_report` to spot per-class weaknesses.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

t0 = time.time()
vec_classical = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_classical = vec_classical.fit_transform(texts_train)
X_test_classical = vec_classical.transform(texts_test)

clf_classical = LogisticRegression(max_iter=1000, random_state=SEED)
clf_classical.fit(X_train_classical, y_train)
y_pred_classical = clf_classical.predict(X_test_classical)
t_classical = time.time() - t0

acc_classical = accuracy_score(y_test, y_pred_classical)
f1_classical = f1_score(y_test, y_pred_classical, average="macro")

print("Classical baseline")
print(f"  accuracy : {acc_classical:.3f}")
print(f"  macro-F1 : {f1_classical:.3f}")
print(f"  time     : {t_classical:.2f}s")
print()
print(classification_report(y_test, y_pred_classical, target_names=["ham", "spam"]))


Classical baseline
  accuracy : 0.971
  macro-F1 : 0.932
  time     : 0.11s

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



## Step 5 — Pretrained transformer (zero-shot)

The slide gives you the recipe (slim slide 7, right column):

```python
from transformers import pipeline

clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

y_pred = [
    1 if clf(t)[0]["label"] == "POSITIVE" else 0
    for t in texts_test
]
```

### Two things to watch out for

1. **The first call downloads ~250 MB** of model weights to your local HF
   cache (`~/.cache/huggingface/`). Be patient.
2. **The model was pre-trained on movie review sentiment**, not SMS spam.
   The slide convention `POSITIVE → 1` was for sentiment. For spam
   detection, `POSITIVE` is semantically *ham* (legit), so you'll need to
   re-map: `NEGATIVE → spam (1)`, `POSITIVE → ham (0)`. The reflection asks
   you to discuss why this domain mismatch matters.

### Your task

- Run the slide code.
- Re-map the predictions so that `1 = spam`, matching the classical
  baseline.
- Compute accuracy, macro-F1, and time.


In [18]:
import time
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

t0 = time.time()

clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt",
    device=-1,
)

results = clf(
    list(texts_test),
    truncation=True,
    max_length=128,
    batch_size=32,
)

y_pred_transformer = [1 if r["label"] == "NEGATIVE" else 0 for r in results]

t_transformer = time.time() - t0

# 5. Metrics
acc_transformer = accuracy_score(y_test, y_pred_transformer)
f1_transformer = f1_score(y_test, y_pred_transformer, average="macro")

print("Pretrained transformer (Cross-Task Transfer)")
print(f"  accuracy : {acc_transformer:.3f}")
print(f"  macro-F1 : {f1_transformer:.3f}")
print(f"  time     : {t_transformer:.2f}s")

Device set to use cpu


Pretrained transformer (Cross-Task Transfer)
  accuracy : 0.451
  macro-F1 : 0.423
  time     : 7.05s


## Step 6 — Compare side by side

Build a DataFrame with one row per model, save it to `results/results.csv`.

### Your task

Create a DataFrame with columns `model`, `accuracy`, `macro_f1`, `time_s`,
one row per approach. Save it.


In [15]:
results = pd.DataFrame([
    {
        "model": "classical",
        "accuracy": acc_classical,
        "macro_f1": f1_classical,
        "time_s": t_classical,
    },
    {
        "model": "transformer",
        "accuracy": acc_transformer,
        "macro_f1": f1_transformer,
        "time_s": t_transformer,
    },
])

results.to_csv(RESULTS_DIR / "results.csv", index=False)
results


,model,accuracy,macro_f1,time_s
0,classical,0.971300,0.931703,0.109772
1,transformer,0.451121,0.423276,6.783730


## Step 7 — Sanity-check the result

A useful gut-check: print 5 messages from the test set with **both**
predictions and the true label. Cases where the two models disagree are the
most informative — that's where the conceptual difference shows up.

### Your task

Build and print a small DataFrame: `text`, `true`, `classical`, `transformer`.
Then filter to rows where the two disagree.


In [16]:
disagree_df = pd.DataFrame({
    "text": texts_test,
    "true": y_test,
    "classical": y_pred_classical,
    "transformer": y_pred_transformer,
})

disagree_df = disagree_df[disagree_df["classical"] != disagree_df["transformer"]]

print(f"Disagreements: {len(disagree_df):,} examples")
disagree_df.head(5)

Disagreements: 628 examples


,text,true,classical,transformer
2,Waiting in e car 4 my mum lor. U leh? Reach ho...,0,0,1
4,If you r @ home then come down within 5 min,0,0,1
5,No need lar. Jus testing e phone card. Dunno n...,0,0,1
6,Wot about on wed nite I am 3 then but only til 9!,0,0,1
7,Hey you gave them your photo when you register...,0,0,1


## Step 8 — Containerise

You need to ship a `Dockerfile` so the grader can reproduce your comparison
with two commands. The minimal version is on slim slide 9 — copy it into
a file called `Dockerfile` (not `Dockerfile.txt`) at the project root.

You also need:
- `requirements.txt` with **pinned** versions of every package you used
- `src/main.py` that runs the same comparison this notebook ran and prints
  the results table to stdout

### Your task

Write the Dockerfile to disk. Then list (here as comments) the entries
your `requirements.txt` will need.


In [19]:
dockerfile = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src/ ./src/
COPY data/ ./data/
CMD ["python", "-m", "src.main"]
"""

Path("Dockerfile").write_text(dockerfile)


179

### Your `requirements.txt` should pin at least

```
numpy==<your-version>
pandas==<your-version>
scikit-learn==<your-version>
transformers==<your-version>
torch==<your-version>
```

You can get the versions with `pip freeze` or `python -m pip list`. Pin
exact versions — `>=` is not pinning.

### Then build and run

```bash
docker build -t nlp-hw .
docker run --rm nlp-hw
```


## Step 9 — Reflection

Write `reflection.md` at the project root. Half a page, three questions.

### 1. Which approach won, and what does that tell you about when a classical method is still worth using?

Compare numbers *and* cost: training time, inference time, model size,
debuggability. When would you actually pick which?

### 2. What's in your Docker image that wouldn't be in a venv, and why does that matter for someone trying to reproduce your work?

Be concrete. Pick one thing (the OS, a system library, the Python
interpreter, the model weights) and explain why it matters.

### 3. One thing that broke, one thing that surprised you.

This one is for you, not for the grade. The pattern you noticed is the
thing you'll remember next year.


## Step 10 — Pre-submission self-check

Before you push the final commit:

- [ ] **Reproducibility:** clone the repo somewhere fresh and run
      `docker build -t nlp-hw . && docker run --rm nlp-hw`. Does it print
      the comparison?
- [ ] **`results/results.csv`** exists and has both models
- [ ] **`requirements.txt`** has pinned versions (no unpinned packages,
      no `>=`)
- [ ] **`.gitignore`** excludes `.venv/`, `__pycache__/`, `data/raw/`,
      `.env`
- [ ] **`README.md`** shows the two reproduce commands at the top
- [ ] **`reflection.md`** answers all three questions, under one page
- [ ] No secrets in the repo (`.env`, API keys, tokens)
- [ ] Final commit is on the `main` branch

Once all eight boxes are ticked, submit the repo URL.
